In [1]:
from spiral import Spiral

sp = Spiral(overrides={
    "keys_cache.enabled": "1",
    "keys_cache.memory_capacity_bytes": "1073741824",  # 1 GiB
    "keys_cache.disk_capacity_bytes": "0",
    "fragments_cache.enabled": "1",
    "fragments_cache.memory_capacity_bytes": "1073741824",  # 1 GiB
    "fragments_cache.disk_capacity_bytes": "0",
})

In [2]:
project = sp.project("enigma-spiral-poc-2-724186")

In [4]:
project.list_tables()

[TableResource(id='table_35vkxf', project_id='enigma-spiral-poc-2-724186', dataset='default', table='stim_constants'),
 TableResource(id='table_3ifkzf', project_id='enigma-spiral-poc-2-724186', dataset='default', table='session_reconstructed_video_metadata'),
 TableResource(id='table_3lzdom', project_id='enigma-spiral-poc-2-724186', dataset='default', table='spike_data'),
 TableResource(id='table_4cpkjo', project_id='enigma-spiral-poc-2-724186', dataset='default', table='vidtok_embeddings'),
 TableResource(id='table_69tszf', project_id='enigma-spiral-poc-2-724186', dataset='default', table='lfp_probe_metadata'),
 TableResource(id='table_78ttpe', project_id='enigma-spiral-poc-2-724186', dataset='default', table='spike_data_by_unit'),
 TableResource(id='table_ekccal', project_id='enigma-spiral-poc-2-724186', dataset='default', table='stim_trial_info'),
 TableResource(id='table_jxze31', project_id='enigma-spiral-poc-2-724186', dataset='default', table='lfp_data'),
 TableResource(id='table

In [5]:
tbl_session_reconstructed_video_metadata = project.table("session_reconstructed_video_metadata")
tbl_spike_data_by_time = project.table("spike_data_by_time")
tbl_vidtok_embeddings = project.table("vidtok_embeddings")
tbl_behavior_adc = project.table("behavior_adc")
tbl_stim_events = project.table("stim_events")

In [6]:
# Random explorations.
tbl_behavior_adc.schema()

Schema({session_id=utf8?, timestamp=i64?, eye_x_px_offset_center=f64?, eye_y_px_offset_center=f64?, neuropixel_sync_in=f64?, photodiode=f64?, pupil_size_in=f64?, reward_input=f64?})

In [7]:
# Random explorations.
tbl_session_reconstructed_video_metadata.to_polars_lazy_frame().head().collect()

session_id,reconstruction_frame_number,bg_color,display,displayed_movie,displayed_movie_frame_number,source_movie,source_movie_frame_number,timestamp,trial_idx,fixation_dot
str,i64,list[f64],str,str,f64,str,f64,i64,i64,struct[4]
"""Goliath_2025-10-20_20-40-05""",0,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52431216,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",1,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52439557,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",2,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52447899,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",3,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52456241,0,"{[255, 0, 0],[0, 0],0,25}"
"""Goliath_2025-10-20_20-40-05""",4,"[127.5, 127.5, 127.5]","""background + fixation""",null,null,null,null,52464582,0,"{[255, 0, 0],[0, 0],0,25}"


In [8]:
import pyarrow as pa
from spiral import Shard
from spiral.core.table import KeyRange  # TODO(marko): Fix this.

def ranges_to_shards(tbl, list_ranges) -> list[Shard]:
    shards = []
    for r in list_ranges:
        st = tbl.key(r["session_id"], r["start"])
        ed = tbl.key(r["session_id"], r["end"])
        key_range = KeyRange(begin=st, end=ed)
        shards.append(Shard(key_range, None))
    return shards

time_ranges = sp.scan({
    "session_id": tbl_session_reconstructed_video_metadata["session_id"],
    "start": tbl_session_reconstructed_video_metadata["timestamp"],
    "end": tbl_session_reconstructed_video_metadata["timestamp"] + 1_000_000,
}).to_table().to_pylist()
sessions_shards = ranges_to_shards(tbl_session_reconstructed_video_metadata, time_ranges)

sessions_shards[0]

Shard { key_range: KeyRange { begin: Key(\x02Goliath\x5f2025\x2d10\x2d20\x5f20\x2d40\x2d05\x00\x18\x03\x20\x09p), end: Key(\x02Goliath\x5f2025\x2d10\x2d20\x5f20\x2d40\x2d05\x00\x18\x03\x2fK\xb0) }, cardinality: None }

In [9]:
from spiral import Sampler

def behavior_sampler_function(array: pa.Array) -> pa.Array:
    return pa.array([i % 10 == 0 for i in range(len(array))])

behavior_sampler = Sampler(behavior_sampler_function)

In [13]:
import tqdm
import numpy as np

embeddings_scan = sp.scan(tbl_vidtok_embeddings["tensor"], where=tbl_vidtok_embeddings["modality"] == "rgb")
behavior_scan = sp.scan(tbl_behavior_adc[["pupil_size_in", "eye_x_px_offset_center", "eye_y_px_offset_center"]])

embeddings_loader = embeddings_scan.to_record_batches(shards=sessions_shards, batch_readahead=64)
behavior_loader = behavior_scan.to_record_batches(shards=sessions_shards, sampler=behavior_sampler, batch_readahead=64)

for behavior, embeddings in tqdm.tqdm(zip(behavior_loader, embeddings_loader)):
    # TODO(marko): Stack here in numpy.
    # print(f"Batch `behavior` {behavior.num_rows} `embeddings` {embeddings.num_rows}")
    # for tensor in embeddings["tensor"]:
    #     print(np.array(tensor).shape)
    # break
    pass

KeyboardInterrupt: 